## Tratamiento datos oferta de precios vivienda (Idalista)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Definición de rutas de ficheros del INE 
export_ine_path = Path(r"C:\Users\LorenaMéndez\Desktop\Analisis_Precios_vivienda\Export_INE")
ccaa_ine_file_path = export_ine_path / "D_CCAA_INE.xlsx"
# Definición de rutas de ficheros del Idealista
precios_oferta_vivienda_idealista = "Export-Idealista-PrecioOfertaVivienda-CCAA.xlsx"

In [3]:
# 1. Cargar fichero de exportación de CCAA del INE
df_ccaa_ine = pd.read_excel(ccaa_ine_file_path, sheet_name="Data")
# Mostrar los resultados
df_ccaa_ine.head(20)

,id,nombre,nombre_completo
0,1,Andalucía,01 Andalucía
1,2,Aragón,02 Aragón
2,3,Asturias,03 Asturias
3,4,Balears,04 Balears
4,5,Canarias,05 Canarias
5,6,Cantabria,06 Cantabria
6,7,Castilla y León,07 Castilla y León
7,8,Castilla - La Mancha,08 Castilla - La Mancha
8,9,Cataluña,09 Cataluña
9,10,Valencia,10 Valencia


In [4]:
# 2. Cargar fichero de exportación del precio de oferta de Idealista

# Cargar todas las pestañas del Excel
dfs = pd.read_excel(precios_oferta_vivienda_idealista, sheet_name=None)

# Mostrar nombres de las pestañas
print("Pestañas encontradas:")
print(dfs.keys())

Pestañas encontradas:
dict_keys(['01 Andalucia', '02 Aragón', '03 Asturias', '04 Balears', '05 Canarias', '06 Cantabria', '07 Castilla y León', '08 Castilla - La Mancha', '09 Cataluña', '10 Com. Valenciana', '11 Extremadura', '12 Galicia', '13 Madrid', '14 Murcia', '15 Navarra', '16 País Vasco', '17 La Rioja', '18 Ceuta', '19 Melilla'])


In [5]:
# Unir todas las pestañas en un único DataFrame
# añadiendo el nombre de la pestaña como columna
df_precio_oferta_vivienda = pd.concat(
    [
        df.assign(ccaa=nombre_hoja)
        for nombre_hoja, df in dfs.items()
    ],
    ignore_index=True
)

# Renombrar nombres de columnas
df_precio_oferta_vivienda = df_precio_oferta_vivienda.rename(columns={"CCAA": "nombre_ccaa"})
df_precio_oferta_vivienda.columns = df_precio_oferta_vivienda.columns.str.lower()
df_precio_oferta_vivienda = df_precio_oferta_vivienda.rename(columns={"precio m2": "precio_m2"})
df_precio_oferta_vivienda = df_precio_oferta_vivienda.rename(columns={"mes": "fecha"})

# Incluir nuevas columnas
df_precio_oferta_vivienda["cod_ccaa"] = df_precio_oferta_vivienda["ccaa"].str[:2].astype(int)
df_precio_oferta_vivienda["mes"] = df_precio_oferta_vivienda["fecha"].dt.month
df_precio_oferta_vivienda["ejercicio"] = df_precio_oferta_vivienda["fecha"].dt.year

# Ordenar las columnas
df_precio_oferta_vivienda = df_precio_oferta_vivienda[['cod_ccaa','nombre_ccaa','medida', 'mes', 'ejercicio','fecha', 'precio_m2']]

# Limpieza de la columna precio_m2
df_precio_oferta_vivienda["precio_m2"] = (
    df_precio_oferta_vivienda["precio_m2"]
    .replace("n.d.", np.nan)
    .str.replace(" €/m2", "", regex=False)
    .str.replace(".", "", regex=False)   # miles
    .str.replace(",", ".")               # decimales
)

df_precio_oferta_vivienda["precio_m2"] = pd.to_numeric(df_precio_oferta_vivienda["precio_m2"], errors="coerce")

# Mostrar los resultados
df_precio_oferta_vivienda

,cod_ccaa,nombre_ccaa,medida,mes,ejercicio,fecha,precio_m2
0,1,Andalucía,Precio Venta,3,2026,2026-03-01,2835.0
1,1,Andalucía,Precio Venta,2,2026,2026-02-01,2817.0
2,1,Andalucía,Precio Venta,1,2026,2026-01-01,2784.0
3,1,Andalucía,Precio Venta,12,2025,2025-12-01,2755.0
4,1,Andalucía,Precio Venta,11,2025,2025-11-01,2735.0
...,...,...,...,...,...,...,...
7997,19,Melilla,Precio Alquiler,10,2013,2013-10-01,9.0
7998,19,Melilla,Precio Alquiler,9,2013,2013-09-01,9.3
7999,19,Melilla,Precio Alquiler,8,2013,2013-08-01,9.2
8000,19,Melilla,Precio Alquiler,7,2013,2013-07-01,9.3


In [6]:
#df_precio_oferta_vivienda.dtypes

In [7]:
# Descarga de los datos a csv y excel
df_precio_oferta_vivienda.to_excel("F_precio_oferta_vivienda_idealista.xlsx", index=False)
df_precio_oferta_vivienda.to_csv("F_precio_oferta_vivienda_idealista.csv", index=False)